# ⚙️ Der Optimizer ist dein Compiler

Traditionell: **Quellcode → Compiler → Binary**
Heute: **Signature + Metrik + Daten → Der Optimizer → Optimierter Prompt**

Beides nimmt menschenlesbare Spezifikationen und produziert maschinenausführbare Artefakte.

Im letzten Notebook hast du gesehen: selbst wenn du dem Modell die gültigen Kategorien, Prioritäten und Gruppennamen gibst, bleibt der Score bei ~30%. Kann ein **automatischer Optimizer** das besser?

## Was passiert hier?

Im letzten Notebook hast du Prompts **manuell** verbessert. Jetzt lässt du den **Computer** das machen.

Das Prinzip ist einfach:
1. Du sagst, **was** du willst (Signature)
2. Du sagst, **was gut heisst** (Metrik)
3. Du gibst **Beispiele** (Daten)
4. Der Optimizer findet automatisch den besten Prompt

Das ist wie ein Compiler: Du schreibst Quellcode, der Compiler macht eine optimierte Binary. Hier schreibst du eine Spezifikation, der Optimizer macht einen optimierten Prompt.


In [ ]:
import sys
sys.path.insert(0, ".")
from dspy_tasks.config import get_available_models, configure_dspy

# Verfügbare Modelle
print("Verfügbare Modelle:")
for m in get_available_models()[:10]:
    print(f"  • {m}")

# Modell wählen (ändere den String um ein anderes zu nutzen)
MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)
print(f"\n✅ Konfiguriert: {MODEL}")

### 🔄 Der Optimizer-Workflow als Diagramm

So sieht automatische Optimierung aus — in vier Schritten. Du lieferst die **Zutaten** (Signature, Metrik, Daten), der Optimizer **kocht** daraus den besten Prompt.

Das Schöne: Wenn sich deine Daten ändern, optimierst du einfach nochmal. Der Prozess ist reproduzierbar.


In [ ]:
from dspy_tasks.visualize import diagram

diagram([
    {"label": "Signature", "detail": "Was du willst", "icon": "📝", "color": "#0078d4"},
    {"label": "Metrik", "detail": "Was 'gut' heisst", "icon": "📐", "color": "#0078d4"},
    {"label": "Trainingsdaten", "detail": "Beispiele", "icon": "📊", "color": "#0078d4"},
    {"label": "Optimizer", "detail": "probiert Varianten", "icon": "⚙️", "color": "#ca5010"},
    {"label": "Optimierter Prompt", "detail": "+ Few-Shot Demos", "icon": "🎯", "color": "#107c10"},
], title="Die Optimierungs-Pipeline")

## ✏️ Erst du, dann die Maschine

Bevor wir den automatischen Optimizer loslassen, versuch es nochmal selbst! Das ist dieselbe Ticket-Routing-Aufgabe aus Notebook 01. 

Editiere den Prompt unten und schau, welchen Score du erreichst. Merk dir deinen besten Score — danach vergleichen wir mit dem automatisch optimierten Ergebnis.

### 🤔 Warum erst manuell?

Gute Frage! Wir wollen, dass du ein **Gefühl** für Prompt-Engineering bekommst. Wenn du selbst versucht hast, einen Prompt zu verbessern, verstehst du viel besser, was der Optimizer tut — und warum er es besser kann als wir.

Merk dir deinen Score — gleich vergleichen wir ihn mit dem des Optimizers.


In [ ]:
from dspy_tasks.actions import run_with_prompt
from dspy_tasks.visualize import display_score, display_results_table

# Dein manueller Prompt — ändere ihn und führe die Zelle erneut aus!
MEIN_PROMPT = """Klassifiziere dieses IT-Support-Ticket.
Kategorie MUSS eine von diesen sein: Event NO Customer Impact, Failure, Service Request
Priorität MUSS eine von diesen sein: High, Medium, Standard
Zuständige Gruppe MUSS eine von diesen sein: SDE - Service Desk, OFC - Office & Collaboration, PRM - Premium Support, CDC - Client Design & Standard SW Integration, OUM - Incident, Helpdesk, Service-Center IKT, Service-Center IKT Bestellungen, Smartcard Office, CBCD - Container Basierte Cloud Dienste, OPC - Applikationen, DevOps - Rein, BVX - ePortal Service Line, IOM - Input / Output Mgmt., OPM - DLC Dispatching, O-SDK, ESTV-RSS-Stammdaten, FIB - BIT Store Bollwerk, Immobilien BAZG, Bedarfsmanagement, Güter und Ausrüstung"""

manual_result = run_with_prompt("ticket_routing", MEIN_PROMPT, max_eval=8)
display_score("Dein manueller Prompt", manual_result.score)
display_results_table(manual_result.individual_scores)

## ⚙️ Und jetzt die Maschine...

Du hast deinen besten manuellen Score gesehen. Selbst MIT allen gültigen Werten im Prompt liegt er vermutlich bei 30-50%. Das Modell kennt die Werte, aber es weiss nicht, **wann welcher Wert passt**.

Jetzt drück unten auf "Optimieren" und schau, was passiert. Der Optimizer lernt aus den Trainingsdaten die **Zuordnungsregeln** und findet automatisch den besten Prompt.

**Die Frage ist:** Kann der Computer einen besseren Prompt finden als du?

## BootstrapFewShot: Der schnelle Compiler

**BootstrapFewShot** ist wie `-O1` Optimierung — schnell und effektiv. Er sucht die besten Few-Shot-Beispiele aus deinen Trainingsdaten und fügt sie in den Prompt ein.

Beim Ticket-Routing heisst das: Der Optimizer wählt automatisch die informativsten Beispiel-Tickets aus, damit das Modell lernt: *"Account gesperrt" → SDE - Service Desk, "CPAM Ausfall" → CDC*.

Dauer: ~10 Sekunden. Verbesserung: oft 10-30%.

### 📋 Was macht BootstrapFewShot genau?

Stell dir vor, du hast 35 Trainings-Tickets. BootstrapFewShot probiert verschiedene Kombinationen durch und findet heraus: *Welche 3-5 Beispiel-Tickets im Prompt liefern die besten Ergebnisse?*

Es ist wie ein Koch, der verschiedene Gewürzkombinationen testet — systematisch statt nach Bauchgefühl.

In [ ]:
from dspy_tasks.actions import run_optimization
from dspy_tasks.tasks import get_task
from dspy_tasks.visualize import display_improvement, display_insight, display_prompt_diff, display_score, display_results_table

task = get_task("ticket_routing")
print(f"⏳ Optimiere {task.name} mit BootstrapFewShot...")
print(f"   Basis: dein manueller Prompt")
print(f"   Das kann 10-60 Sekunden dauern...\n")

bs_result = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=8, instructions=MEIN_PROMPT)

print("━" * 60)
print("📊 NACHHER: Optimierter Prompt (dein Prompt + Few-Shot Beispiele)")
display_score("Nach BootstrapFewShot", bs_result.optimized_score)
if bs_result.optimized_individual_scores:
    display_results_table(bs_result.optimized_individual_scores)

display_improvement(manual_result.score, bs_result.optimized_score)

display_prompt_diff(MEIN_PROMPT, bs_result.prompt_after)

display_insight("Was gerade passiert ist",
    f"Dein Prompt: {manual_result.score:.0%} → BootstrapFewShot: {bs_result.optimized_score:.0%}. "
    "Der Optimizer hat deinen Prompt als Basis genommen und die besten Beispiel-Tickets hinzugefügt.")

## MIPROv2: Der Heavy-Duty Compiler

**MIPROv2** ist wie `-O3` — er optimiert gleichzeitig die **Instruktionen UND die Beispiele** mit Bayesian Search. Während BootstrapFewShot nur Beispiele auswählt, schreibt MIPROv2 den Prompt-Text selbst um.

Dauer: ~30-60 Sekunden. Verbesserung: oft nochmal besser als BootstrapFewShot.

### 🔀 BootstrapFewShot vs. MIPROv2 — der direkte Vergleich

Zwei verschiedene Strategien treten gegeneinander an:

| Optimizer | Was er optimiert | Stärke |
|---|---|---|
| **BootstrapFewShot** | Welche Beispiele im Prompt stehen | Schnell, zuverlässig |
| **MIPROv2** | Beispiele UND Anweisungen | Gründlicher, aber langsamer |

Welcher gewinnt? Klick den Button und finde es heraus. Spoiler: Es hängt vom Task ab!


In [ ]:
from dspy_tasks.visualize import bar_comparison

task = get_task("ticket_routing")

# --- BootstrapFewShot (basierend auf manuellem Prompt) ---
print(f"⏳ BootstrapFewShot auf {task.name}...\n")
r_bs = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=8, instructions=MEIN_PROMPT)
print(f"✅ BootstrapFewShot: {r_bs.optimized_score:.0%}")

print("\n📝 BootstrapFewShot — Prompt:")
print(r_bs.prompt_after[:2000])
if len(r_bs.prompt_after) > 2000:
    print(f"... ({len(r_bs.prompt_after)} Zeichen)")

print("\n📊 BootstrapFewShot — Ergebnisse pro Ticket:")
if r_bs.optimized_individual_scores:
    display_results_table(r_bs.optimized_individual_scores)

# --- MIPROv2 (basierend auf dem BESSEREN der beiden bisherigen Ergebnisse) ---
print(f"\n{'━'*60}")
best_score = max(manual_result.score, r_bs.optimized_score)
best_prompt = MEIN_PROMPT if manual_result.score >= r_bs.optimized_score else r_bs.prompt_after
best_label = "Manuell" if manual_result.score >= r_bs.optimized_score else "BootstrapFewShot"
print(f"⏳ MIPROv2 auf {task.name}...")
print(f"   Basis: {best_label} ({best_score:.0%}) — das bisherige beste Ergebnis\n")

r_mipro = run_optimization("ticket_routing", "MIPROv2", max_eval=8, instructions=best_prompt)
print(f"✅ MIPROv2: {r_mipro.optimized_score:.0%}")

print("\n📝 MIPROv2 — Prompt:")
print(r_mipro.prompt_after[:2000])
if len(r_mipro.prompt_after) > 2000:
    print(f"... ({len(r_mipro.prompt_after)} Zeichen)")

print("\n📊 MIPROv2 — Ergebnisse pro Ticket:")
if r_mipro.optimized_individual_scores:
    display_results_table(r_mipro.optimized_individual_scores)

# --- Vergleich aller 3 ---
print(f"\n{'━'*60}")
print("📊 Alle 3 Prompts im Vergleich:")
print(f"   1. Manueller Prompt:   {manual_result.score:.0%}  (deine Anweisung)")
print(f"   2. BootstrapFewShot:   {r_bs.optimized_score:.0%}  (dein Prompt + Beispiele)")
print(f"   3. MIPROv2:            {r_mipro.optimized_score:.0%}  (bester Prompt + Beispiele + umgeschrieben)")

scores = {
    "Manuell": {"baseline": manual_result.score, "optimized": manual_result.score},
    "BootstrapFewShot": {"baseline": manual_result.score, "optimized": r_bs.optimized_score},
    "MIPROv2": {"baseline": manual_result.score, "optimized": r_mipro.optimized_score},
}
fig = bar_comparison("Ticket Routing: Manuell vs. Optimizer", scores)
fig.show()

## 🔬 Warum bleibt der Score unter 100%? — Daten-Analyse

Bevor wir raten, schauen wir uns die **Daten** an. Drei Visualisierungen zeigen, warum das Modell Schwierigkeiten hat:

In [ ]:
import json, math
from collections import Counter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

with open("datasets/ticket_routing.json") as f:
    tickets = json.load(f)

# ━━━ 1. Datenverteilung: Wie viele Beispiele pro Klasse? ━━━
groups = Counter(t["assigned_group"] for t in tickets)
cats = Counter(t["category"] for t in tickets)
prios = Counter(t["priority"] for t in tickets)

# Gruppen-Verteilung (das Kernproblem)
sorted_groups = groups.most_common()
names = [g[0] for g in sorted_groups]
counts = [g[1] for g in sorted_groups]
colors = ["#2ea043" if c >= 4 else "#ca5010" if c >= 2 else "#f85149" for c in counts]

fig = go.Figure(go.Bar(
    x=counts, y=names, orientation='h',
    marker_color=colors,
    text=[f"{c} {'Beispiel' if c == 1 else 'Beispiele'}" for c in counts],
    textposition='outside'
))
fig.update_layout(
    title="Training-Beispiele pro Gruppe — das Kernproblem",
    xaxis_title="Anzahl Trainings-Tickets",
    yaxis=dict(autorange="reversed"),
    height=600, margin=dict(l=250),
    annotations=[dict(
        text=f"🔴 {sum(1 for c in counts if c == 1)} von {len(counts)} Gruppen haben nur 1 Beispiel!",
        xref="paper", yref="paper", x=0.95, y=0.05,
        showarrow=False, font=dict(size=14, color="#f85149"),
        bgcolor="rgba(248,81,73,0.1)", borderpad=8
    )]
)
fig.show()

# ━━━ 2. Entropy pro Feld: Wie "zufällig" sind die Labels? ━━━
def entropy(counter):
    total = sum(counter.values())
    if total == 0: return 0
    probs = [c / total for c in counter.values()]
    return -sum(p * math.log2(p) for p in probs if p > 0)

def max_entropy(n_classes):
    return math.log2(n_classes) if n_classes > 0 else 0

fields = {
    "Category": (cats, 3),
    "Priority": (prios, 3),
    "Assigned Group": (groups, 21),
}

print("━" * 60)
print("📊 Entropy-Analyse: Wie schwer ist jedes Feld?")
print("━" * 60)
for name, (counter, n_classes) in fields.items():
    ent = entropy(counter)
    max_ent = max_entropy(n_classes)
    normalized = ent / max_ent if max_ent > 0 else 0
    bar = "█" * int(normalized * 20) + "░" * (20 - int(normalized * 20))
    print(f"\n  {name}:")
    print(f"    Klassen: {n_classes} | Entropy: {ent:.2f} / {max_ent:.2f} (max)")
    print(f"    Schwierigkeit: [{bar}] {normalized:.0%}")
    print(f"    Verteilung: {dict(counter.most_common(5))}")

n_single = sum(1 for c in groups.values() if c == 1)
n_groups = len(groups)
top_cat = cats.most_common(1)[0]
print(f"\n💡 Entropy misst die 'Zufälligkeit' der Labels.")
print(f"   Category hat nur 3 Werte, {top_cat[1]}/{sum(cats.values())} davon '{top_cat[0]}' → relativ leicht.")
print(f"   Assigned Group hat {n_groups} Werte, {n_single} davon mit nur 1 Beispiel → Schwierigkeit {entropy(groups)/max_entropy(n_groups):.0%}.")

# ━━━ 3. Self-Consistency: Wie stabil sind die Vorhersagen? ━━━
print(f"\n{'━'*60}")
print("🎲 Self-Consistency: Ist das Modell sich sicher?")
print("   Wir fragen das Modell 3x dasselbe — stimmen die Antworten überein?")
print("━" * 60)

import dspy
from dspy_tasks.tasks import get_task
task = get_task("ticket_routing")
examples = task.load_examples()[:5]  # Test-Beispiele
module = task.make_module()

for ex in examples[:3]:
    summary = str(ex.summary)[:60]
    predictions = []
    for run in range(3):
        try:
            # temperature > 0 erzwingt Variation
            with dspy.context(temperature=0.7):
                pred = module(summary=ex.summary)
            predictions.append({
                "category": str(getattr(pred, 'category', '?')),
                "priority": str(getattr(pred, 'priority', '?')),
                "assigned_group": str(getattr(pred, 'assigned_group', '?')),
            })
        except:
            predictions.append({"category": "ERROR", "priority": "ERROR", "assigned_group": "ERROR"})
    
    print(f"\n  📝 '{summary}...'")
    for field in ["category", "priority", "assigned_group"]:
        values = [p[field] for p in predictions]
        unique = len(set(values))
        agreement = values.count(values[0]) / len(values)
        icon = "✅" if unique == 1 else "⚠️" if unique == 2 else "🔴"
        print(f"    {icon} {field}: {' | '.join(values)}")
        if unique > 1:
            print(f"       → {unique} verschiedene Antworten! Agreement: {agreement:.0%}")

print(f"\n💡 Bei niedrigem Agreement 'rät' das Modell — es hat keine klare Regel.")
print(f"   Genau diese Fälle profitieren am meisten von mehr Trainingsdaten.")

### 🎯 Was die Analyse zeigt

Die drei Visualisierungen oben enthüllen das **echte Problem**:

| Beobachtung | Was es bedeutet |
|---|---|
| **Die Daten sind relativ gut verteilt** (18 von 20 Gruppen haben 4 Beispiele) | Kein 1-Beispiel-Problem — aber trotzdem nicht genug für 20 Klassen |
| **Category-Entropy bei 76%** (58% = Service Request, 36% = Failure) | Ungleich verteilt, aber nicht trivial — "Event NO Customer Impact" hat nur 4 Beispiele |
| **Priority-Entropy bei 71%** (69% = Standard) | "Standard" dominiert — "High" hat nur 5 von 77 Beispielen |
| **Group-Entropy bei 98%** — fast maximal | 20 Klassen, fast gleichverteilt — das härteste Feld, weil das Modell zwischen 20 ähnlichen Optionen wählen muss |
| **Self-Consistency ist hoch** (3/3 Übereinstimmung) | Das Modell ist sich **sicher** — aber sicher **falsch**: es erfindet eigene Kategorien wie "Access" oder "Hardware" statt die gültigen Werte zu nutzen |

**Das zentrale Problem:** Das Modell antwortet konsistent, aber mit **eigenen Kategorien** ("Access", "Hardware", "Workplace Support / ...") statt den vorgegebenen Werten ("Service Request", "Failure", "SDE - Service Desk", ...). Es hat eine klare interne Logik — aber die stimmt nicht mit den Domain-spezifischen Labels überein.

**Techniken aus der Forschung:**

1. **Logprobs** — Die internen Wahrscheinlichkeiten pro Token zeigen, wie "sicher" das Modell bei jedem Wort ist. `exp(logprob)` = Confidence 0-1. (Braucht API-Zugang zu Token-Probabilities)
2. **Self-Consistency Sampling** — Dasselbe Prompt N-mal mit `temperature > 0` ausführen und Übereinstimmung messen. Hohe Übereinstimmung = hohes Vertrauen. (Was wir oben gemacht haben)
3. **Calibration** — Messen ob ein Modell das 90% Confidence zuweist auch wirklich zu 90% richtig liegt. LLMs sind typischerweise *overconfident*.
4. **SelfCheckGPT** — Mehrere Antworten generieren und prüfen ob sie sich widersprechen. Widersprüche = Halluzination/Unsicherheit.

> **Die Kernfrage ist immer:** Wann kann ich dem Modell vertrauen? Die Antwort: Hohe Self-Consistency allein reicht nicht — das Modell kann konsistent falsch liegen. Es braucht **Trainingsdaten mit den richtigen Labels**, damit der Optimizer dem Modell beibringen kann, welche Domain-spezifischen Werte erwartet werden.

## 🏆 Vergleich: Du vs. Maschine

Scroll zurück und vergleich die Ergebnistabellen:
- **Dein manueller Prompt** (oben): Schau dir an, welche Tickets du richtig hattest
- **BootstrapFewShot**: Hat er die gleichen Fehler gemacht? Oder andere?
- **MIPROv2**: Wo hat der umgeschriebene Prompt geholfen?

Der Unterschied zeigt sich in den **Details**:
- Der Optimizer wählt Beispiel-Tickets, die dem Modell die **Zuordnungsregeln** beibringen
- MIPROv2 schreibt zusätzlich die Anweisungen um — oft in einer Art, die du selbst nicht probiert hättest
- Aber beide Optimizer können nur Muster finden, die **in den Trainingsdaten vorhanden sind**

> **Das ist der Punkt:** Manuelles Prompt-Tuning funktioniert, ist aber langsam und fragil. Automatische Optimierung findet bessere Prompts, schneller, reproduzierbar. **Deine Metrik + Daten = dein Programm. Der Optimizer ist der Compiler.**

## ⏭️ Weiter geht's!

Du hast gesehen: der Optimizer findet bessere Prompts als du — automatisch, messbar, reproduzierbar.

Im nächsten Notebook testen wir das auf **verschiedenen Aufgaben** und mit **deinen echten Daten**. Dort zeigt sich der wahre Wert: **Domain-Tuning = Burggraben.**

👉 **[Weiter zu Notebook: Domain-Tuning →](03_domain_tuning.ipynb)**
